# Python and CLI for Data Science - Session 16

- *Course*: Python and CLI for Data Science
- *Session*: 16
- *Unit*: Scikit-Learn - Regression

### (Re)sources:
- [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/index.html) _by Jake VanderPlas (Code released under the MIT License)_


- This section will take a closer took at some regression algorithms
    
    *Regression is the task of predicting a continuous quantity*
       

- We will focus on three of the most common regression algorithms
    - Linear Regression
    - Random Forest
    - Support Vector Machines

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

# Linear Regression

- Linear regression models are popular because they are efficient and straightforward to interpret
- The simplest form of linear regression is fitting a straight line
- The model can be extended to fit more complex data

## Simple Linear Regression

- A straight-line fit is a model of the form where $a$ is commonly known as the *slope*, and $b$ is commonly known as the *intercept*
$$
y = ax + b
$$


- For example, we could have data scattered around a line with a slope of 2 and an intercept of –5:

In [ ]:
rng = np.random.RandomState(1)
x = 10 * rng.rand(50)
y = 2 * x - 5 + rng.randn(50)
plt.scatter(x, y);

- Scikit-Learn's `LinearRegression` estimator can fit this data

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression(fit_intercept=True)
model.fit(x[:, np.newaxis], y)

xfit = np.linspace(0, 10, 1000)
yfit = model.predict(xfit[:, np.newaxis])

plt.scatter(x, y)
plt.plot(xfit, yfit);

- The slope and intercept of the data are contained in the model's fit parameters as `coef_` and `intercept_`
- The values are very close to the values used to generate the data

In [ ]:
print("Model slope:    ", model.coef_[0])
print("Model intercept:", model.intercept_)

- In addition to simple straight-line fits, linear regression can also handle multidimensional linear models of the form:
$$
y = a_0 + a_1 x_1 + a_2 x_2 + \cdots
$$

- Geometrically, this is akin to fitting a plane to points in three dimensions, or fitting a hyperplane to points in higher dimensions
- The multidimensional nature of such regressions makes them more difficult to visualize

- However, we can see one of these fits in action by building some example data
- Here the $y$ data is constructed from a linear combination of three random $x$ values, and the linear regression recovers the coefficients used to construct the data

In [ ]:
rng = np.random.RandomState(1)
X = 10 * rng.rand(100, 3)
y = 0.5 + np.dot(X, [1.5, -2., 1.])

model.fit(X, y)
print(model.intercept_)
print(model.coef_)

## Basis Function Regression

- Despite it's name, linear regression is not restricted to fitting *lines*
- By transforming the data according to *basis functions* we can adapt linear regression to nonlinear relationships between variables

- The idea is to take our multidimensional linear model and build extra features $x_1, x_2, x_3, \dots$ based on a single feature $x$
- That is, we let $x_n = f_n(x)$, where $f_n()$ is some function that transforms our data

- For example, if $f_n(x) = x^n$, our model becomes a polynomial regression:
$$
y = a_0 + a_1 x + a_2 x^2 + a_3 x^3 + \cdots
$$

- This is *still a linear model*—the linearity refers to the fact that the coefficients $a_n$ never multiply or divide each other
- We have effectively projected our one-dimensional $x$ values into a higher dimension
- This allows a linear fit to capture more complicated relationships between $x$ and $y$

### Polynomial Basis Functions

- This polynomial projection is useful enough that it is built into Scikit-Learn using the `PolynomialFeatures` transformer
- The transformer converts our one-dimensional array into a three-dimensional array, where each column contains the exponentiated value

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(3, include_bias=False)
x = np.array([2, 3, 4])
poly.fit_transform(x[:, None])

- This new, higher-dimensional data representation can then be plugged into a linear regression
- The cleanest way to accomplish this is to use a pipeline
- Let's use a polynomial features up to a degree of 7

In [ ]:
from sklearn.pipeline import make_pipeline
poly_model = make_pipeline(
    PolynomialFeatures(7),
    LinearRegression(),
)

- We can use the linear model to fit much more complicated relationships between $x$ and $y$
- For example, here is a sine wave with noise

In [ ]:
rng = np.random.RandomState(1)
x = 10 * rng.rand(50)
y = np.sin(x) + 0.1 * rng.randn(50)

fig, ax = plt.subplots()
ax.scatter(x, y);

Using our `LinearRegression` model with polynomial basis functions, we can fit this curve

In [ ]:
poly_model.fit(x[:, np.newaxis], y)

xfit = np.linspace(0, 10, 1000)
yfit = poly_model.predict(xfit[:, np.newaxis])

fig, ax = plt.subplots()
ax.scatter(x, y)
ax.plot(xfit, yfit);

- Note that the fit is just a linear combination of the polynomial basis functions
- Despite the complicated appearance of the plot, this is still a linear model! 
- If we plot the individual basis functions together with the fitted model, we get a better idea of what the model is doing

In [ ]:
fig, ax = plt.subplots()

# fit and predict model
poly_model.fit(x[:, np.newaxis], y)
xfit = np.linspace(0, 10, 1000)
yfit = poly_model.predict(xfit[:, np.newaxis])

# plot data and model prediction
ax.scatter(x, y)
ax.plot(xfit, yfit)
y_lim = ax.get_ylim() # save the y-limit for later

# grab the learned coefficients and intercept
coeffs = poly_model.named_steps["linearregression"].coef_
intercept = poly_model.named_steps["linearregression"].intercept_

# iterate over the 7 polynomial basis functions
polyns = []
for exp, coeff in enumerate(coeffs):
    # compute the basis function
    polyn = xfit**exp * coeff
    # store it for later
    polyns.append(polyn)
    # plot the basis function
    ax.plot(xfit, polyn, label=str(exp))

# plot the sum of all basis functions and the intercept
ax.plot(
    xfit,
    np.sum(polyns, axis=0) + intercept,
    color="gray",
    linestyle=":",
    linewidth=4,
)
ax.legend(ncols=3)
ax.set_ylim(y_lim);

**Question**

- What happens if we let the model predict values outside of the interval it was trained on?
- Will the model continue to predict the sinusoidal curve or will it diverge?

In [ ]:
xfit = np.linspace(-1.5, 11.5, 1000) # change the start and end values
yfit = poly_model.predict(xfit[:, np.newaxis])

fig, ax = plt.subplots()
ax.scatter(x, y)
ax.plot(xfit, yfit);

## Exercise

- Train a linear regression model to predict the number of bicycle trips across Seattle's Fremont Bridge based on weather, season, and other factors
- We first download and preprocess the data

In [ ]:
import pandas as pd
import os

!curl -O https://raw.githubusercontent.com/jakevdp/bicycle-data/main/FremontBridge.csv
!curl -O https://raw.githubusercontent.com/jakevdp/bicycle-data/main/SeattleWeather.csv

counts = pd.read_csv(
    "FremontBridge.csv", index_col="Date", parse_dates=True
)
weather = pd.read_csv(
    "SeattleWeather.csv", index_col="DATE", parse_dates=True
)

- The counts DataFrame contains the number of bicycle crossings of the bridge per hour

In [ ]:
counts.head()

- The weather data includes several columns of temperature data, as well as information about the amount of precipitation, snow, and so on

In [ ]:
weather.head()

- For simplicity, we will remove data before 2020 in order to avoid the effects of the COVID-19 pandemic

In [ ]:
counts = counts[counts.index < "2020-01-01"]
weather = weather[weather.index < "2020-01-01"]

- Next we will compute the total daily bicycle traffic, and put this in its own `DataFrame`

In [ ]:
daily = counts.resample('d').sum()
daily['Total'] = daily.sum(axis=1)
daily = daily[['Total']] # remove other columns
daily

- Let's take a look at the data

In [ ]:
daily[['Total']].plot(alpha=0.5);

- We can assume the daily bike commuters vary from day to day
- We account for this in our data by adding one-hot-encoded column for days of the week

In [ ]:
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
for i in range(7):
    daily[days[i]] = (daily.index.dayofweek == i).astype(float)
daily

- Similarly, riders might behave differently on holidays; let's add an indicator of this as well

In [ ]:
from pandas.tseries.holiday import USFederalHolidayCalendar
cal = USFederalHolidayCalendar()
holidays = cal.holidays('2012', '2020')
daily = daily.join(pd.Series(1, index=holidays, name='holiday'))
daily['holiday'].fillna(0, inplace=True)
daily

- Also, the hours of daylight will most likely effect how many people ride their bike
- Let's use the standard astronomical calculation to add this information

In [ ]:
def hours_of_daylight(date, axis=23.44, latitude=47.61):
    """Compute the hours of daylight for the given date"""
    from datetime import datetime
    days = (date - datetime(2000, 12, 21)).days
    m = (1. - np.tan(np.radians(latitude))
         * np.tan(np.radians(axis) * np.cos(days * 2 * np.pi / 365.25)))
    return 24. * np.degrees(np.arccos(1 - np.clip(m, 0, 2))) / 180.

daily['daylight_hrs'] = list(map(hours_of_daylight, daily.index))
daily[['daylight_hrs']].plot()
plt.ylim(8, 17);
# daily

- We can also add the average temperature and total precipitation to the data
- In addition to the inches of precipitation, let's add a flag that indicates whether a day is dry (has zero precipitation):

In [ ]:
weather['Temp (F)'] = 0.5 * (weather['TMIN'] + weather['TMAX'])
weather['Rainfall (in)'] = weather['PRCP']
weather['dry day'] = (weather['PRCP'] == 0).astype(int)

daily = daily.join(weather[['Rainfall (in)', 'Temp (F)', 'dry day']])
daily

- Finally, we add a counter that increases from day 1, and measures how many years have passed
- This will let us measure any observed annual increase or decrease in daily crossings

In [ ]:
daily['annual'] = (daily.index - daily.index[0]).days / 365.
daily

- Now our data is in order, and we can take a look at it

In [ ]:
# Drop any rows with null values
daily = daily.dropna(axis=0, how="any")
daily.head()

- With the data in place we can fit a model
- We plot our predicted numbers versus the actual numbers to gauge how well our model predicts the numbers
- We also use the mean absolute difference to asses how well our model does


- **Test out different features to see which combinations work best**

In [ ]:
from sklearn.metrics import mean_absolute_error

column_names = [
    "Mon",
    "Tue",
    "Wed",
    "Thu",
    "Fri",
    "Sat",
    "Sun",
    "holiday",
    "daylight_hrs",
    "Rainfall (in)",
    "dry day",
    "Temp (F)",
    "annual",
]
X = daily[column_names]
y = daily["Total"]
X_train = X[X.index < "2019-01-01"]
y_train = y[y.index < "2019-01-01"]
X_test = X[X.index >= "2019-01-01"]
y_test = y[y.index >= "2019-01-01"]

model = LinearRegression(fit_intercept=True)
model.fit(X_train, y_train)
y_pred = pd.Series(model.predict(X_test), index=X_test.index)

print(mean_absolute_error(y_test, y_pred))
plt.plot(y_test.index, y_test.values, alpha=0.5, label="actual")
plt.plot(y_pred.index, y_pred.values, alpha=0.5, label="predicted")
plt.legend();

## Interpreting the model

- We can also interpret the model's coefficients to gain insights into the effect of a feature on the number of bike riders
- Additionally, using bootstrap resampling we can add a measure of uncertainty (details into how this works are beyond the scope of this section)

- The table shows per feature how much it affects number of bike riders
- For example, for every hour of daylight, the number of bike riders increases by the value in the effect column ± the uncertainty value

In [ ]:
from sklearn.utils import resample

np.random.seed(1)
err = np.std([model.fit(*resample(X, y)).coef_ for i in range(1000)], 0)
params = pd.Series(model.coef_, index=X.columns)
print(model.intercept_)
pd.DataFrame({"effect": params.round(0), "uncertainty": err.round(0)})

## Random Forest Regression

- We've seen how to fit a simple Decision Tree model to classify data into discrete classes
- Decision trees can also be applied to regression problems
- The `DecisionTreeRegressor` predicts a single value instead of a class label in the leaf of a node
- The result is a piecewise-constant approximation to the data

- Similar to classification, decision tree regressors are prone to overfitting
- Take the following example, where at a depth of five, the decision tree regressor is clearly overfitting the data

In [ ]:
from sklearn.tree import DecisionTreeRegressor

rng = np.random.RandomState(1)
X = np.sort(5 * rng.rand(80, 1), axis=0)
y = np.sin(X).ravel()
y[::5] += 3 * (0.5 - rng.rand(16))

_, ax = plt.subplots()

ax.scatter(X, y, s=20, c="black")
for max_depth in (2, 5):
    reg = DecisionTreeRegressor(max_depth=max_depth)
    reg.fit(X, y)
    Xfit = np.arange(0.0, 5.0, 0.01)[:, np.newaxis]
    xfit = reg.predict(Xfit)
    ax.plot(
        Xfit, xfit, label=f"max_depth={max_depth}", linewidth=2
    )

ax.legend()

- Again, we can counteract overfitting by combining multiple decision trees into an ensemble model, called a *random forest*

In [ ]:
from sklearn.ensemble import RandomForestRegressor

_, ax = plt.subplots()

ax.scatter(X, y, s=20, c="black")
reg = RandomForestRegressor(
    n_estimators=1000,
    max_depth=5,
    random_state=1,
)
reg.fit(X, y)
Xfit = np.arange(0.0, 5.0, 0.01)[:, np.newaxis]
xfit = reg.predict(Xfit)
ax.plot(Xfit, xfit, linewidth=2)

## Exercise

- Switch out the `LinearRegression` model for a `RandomForestRegressor` to predict the number of bike riders
- Test out different hyperparameters and try to get the lowest possible mean absolute error
- Is the `RandomForestRegressor` better or worse than the `LinearRegression` model and why?

In [ ]:
column_names = [
    "Mon",
    "Tue",
    "Wed",
    "Thu",
    "Fri",
    "Sat",
    "Sun",
    "holiday",
    "daylight_hrs",
    "Rainfall (in)",
    "dry day",
    "Temp (F)",
    "annual",
]
X = daily[column_names]
y = daily["Total"]
X_train = X[X.index < "2019-01-01"]
y_train = y[y.index < "2019-01-01"]
X_test = X[X.index >= "2019-01-01"]
y_test = y[y.index >= "2019-01-01"]

# model = LinearRegression(fit_intercept=True)
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=100,
    random_state=2,
)
model.fit(X, y)
y_pred = pd.Series(model.predict(X_test), index=X_test.index)

print(mean_absolute_error(y_test, y_pred))
plt.plot(y_test.index, y_test.values, alpha=0.5, label="actual")
plt.plot(y_pred.index, y_pred.values, alpha=0.5, label="predicted")
plt.legend();

## Support Vector Regression

- Support vector machines can also be used for regression


- In classification, the model maximizes the margin between two classes and the support vectors are the data points that are closest to the margin
- The support vectors define the decision boundary


- In regression the model minimizes the distance between the support vectors and the regression line, given a fixed margin or tolerance
- The support vectors are the data points that lie on or outside the tolerance


In [ ]:
def plot_svr_function(model, ax=None):
    if ax is None:
        ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    ax.scatter(
        X[model.support_, 0],
        y[model.support_],
        s=300,
        linewidth=1,
        edgecolors="black",
        facecolors="none",
    )

    Xfit = np.linspace(xlim[0], xlim[1], 1000)[:, None]
    yfit = model.predict(Xfit)

    for vector_idx in model.support_:
        x = X[vector_idx, 0]
        y1 = y[vector_idx]
        y2 = model.predict([[x]])[0]
        ax.plot([x, x], [y1, y2], color="red")

        
    ax.plot(Xfit, yfit, color="black")
    ax.plot(Xfit, yfit + model.epsilon, color="black", linestyle=":")
    ax.plot(Xfit, yfit - model.epsilon, color="black", linestyle=":")

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

In [ ]:
from sklearn.svm import SVR

rng = np.random.RandomState(1)
X = np.sort(5 * rng.rand(40, 1), axis=0)
y = 2 + 1.5 * X.ravel() + rng.randn(X.shape[0])

model = SVR(kernel="linear", epsilon=1)
model.fit(X, y)

fig, ax = plt.subplots()
ax.scatter(X, y, color="black")

xfit = np.linspace(0, 5, 1000)
yfit = model.predict(xfit[:, None])

ax.plot(xfit, yfit, color="black")
plot_svr_function(model)

- The support vectors are chosen based on the tolerance hyperparameter epsilon
- With a higher/lower epsilon, the margin around the regression line increases/decreases

In [ ]:
model = SVR(kernel="linear", epsilon=2)
model.fit(X, y)

fig, ax = plt.subplots()
ax.scatter(X, y, color="black")
xfit = np.linspace(0, 5, 1000)
yfit = model.predict(xfit[:, None])

ax.plot(xfit, yfit, color="black")
plot_svr_function(model)

### Fitting non-linear data

- Support vector regression also makes use of the kernel trick to fit non-linear data
- Consider the following sinusoidal data
- A linear support vector regressor does not fit the data well

In [ ]:
rng = np.random.RandomState(1)
X = np.sort(5 * rng.rand(40, 1), axis=0)
y = np.sin(X).ravel()

y[::5] += 3 * (0.5 - rng.rand(8))

model = SVR(kernel="linear", epsilon=1)
model.fit(X, y)

Xfit = np.linspace(0, 5, 100)[:, None]
yfit = model.predict(Xfit)

fig, ax = plt.subplots()
ax.plot(X, y, "o", color="black")

plot_svr_function(model, ax=ax)

- By changing the kernel to a radial basis function (RBF) kernel, we can model non-linear relationships

In [ ]:
model = SVR(kernel="rbf", epsilon=1)
model.fit(X, y)
  
Xfit = np.linspace(0, 5, 100)[:, None]
yfit = model.predict(Xfit)

fig, ax = plt.subplots()
ax.plot(X, y, "o", color="black")

plot_svr_function(model, ax=ax)

- Let's take a look at 3 different kernel options and how they fit to the data
- The linear kernel is unable to fit to the non-linear data
- The rbf kernel fits to the data almost perfectly
- The polynomial kernel improves over the linear kernel but 

In [ ]:
kernels = ["linear", "rbf", "poly"]
fig, ax = plt.subplots()

ax.scatter(X, y, color="black")
for ix, kernel in enumerate(kernels):
    svr = SVR(kernel=kernel)
    ax.plot(
        Xfit,
        svr.fit(X, y).predict(Xfit),
        lw=2,
        label=f"{kernel} model",
    )

ax.legend();

## Exercise

- Test the `SVR` regression model for predicting the number of bike riders
- Test out different hyperparameters and try to get the lowest possible mean absolute error
- Which model, `LinearRegression`, `RandomForestRegressor`, or `SVR` works best and why?

In [ ]:
column_names = [
    "Mon",
    "Tue",
    "Wed",
    "Thu",
    "Fri",
    "Sat",
    "Sun",
    "holiday",
    "daylight_hrs",
    "Rainfall (in)",
    "dry day",
    "Temp (F)",
    "annual",
]
X = daily[column_names]
y = daily["Total"]
X_train = X[X.index < "2019-01-01"]
y_train = y[y.index < "2019-01-01"]
X_test = X[X.index >= "2019-01-01"]
y_test = y[y.index >= "2019-01-01"]

# model = LinearRegression(fit_intercept=True)
model = RandomForestRegressor(
    n_estimators=1000,
    max_depth=100,
    random_state=1,
)
# model = SVR(kernel="linear", epsilon=0.1, C=1)
model.fit(X, y)
y_pred = pd.Series(model.predict(X_test), index=X_test.index)

print(mean_absolute_error(y_test, y_pred))
plt.plot(y_test.index, y_test.values, alpha=0.5, label="actual")
plt.plot(y_pred.index, y_pred.values, alpha=0.5, label="predicted")
plt.legend();